# LLMScholar-Personas — Metrics Pipeline

Implements the exact metrics from LLMScholarBench (Espín-Noboa & Méndez, 2026):
- **Diversity** (normalized Shannon entropy, Eq. 9)
- **Parity** (1 − TV distance, Eq. 11)
- **Factuality** (matched / unique names, Eq. 5)
- **Consistency** (pairwise Jaccard across runs, Eq. 4)
- **Duplicates** (1 − unique/total, Eq. 3)

**Known gaps (flagged):**
- `created_at` is NaN in factuality_full → using `run_id` for ordering
- Author language not available → `diversity_language` skipped
- Gender and geography only available for *found* authors

## Step 0 — Setup and data loading

In [ ]:
from pathlib import Path
import sys
import glob
import hashlib
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── Paper style: pulled from libs so all notebooks share one source of truth ──
if '../..' not in sys.path:
    sys.path.append('../..')
from libs.visuals.vis import sns_paper_style
from libs.visuals.grouped_metrics import plot_grouped_metrics
from libs.metrics.constants import FIG_DPI

sns_paper_style(font_scale=1.55)

RESULTS      = Path('../../../results')
FACT_PATH    = RESULTS / 'summary/factuality_full.csv'
ETH_GT_GLOB  = str(RESULTS / 'ethnicity/DataFrameRankings_Genderize_Namsor_*_with_ethnicity.csv')
SS_GT_PATH   = Path('/data/datasets/LLMScholar-Personas/data/semantic_scholar_data/clean/Researchers_Deduplicated_Genderize_Namsor.parquet')
FIG_DIR  = RESULTS / 'figures'
PDF_DIR  = RESULTS / 'plots_pdf'
CACHE_DIR = RESULTS / '.cache'
FIG_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CALL_KEYS   = ['model', 'role', 'task', 'location', 'k', 'target', 'field', 'subfield', 'language', 'run_id']
PROMPT_KEYS = [c for c in CALL_KEYS if c != 'run_id']

VALID_FLAGS = {'unchanged', 'cleaned', 'fixed_dict'}


def _file_hash(*paths) -> str:
    """MD5 of mtime+size for each path — changes when any source file is updated."""
    parts = []
    for p in paths:
        p = Path(p)
        if p.exists():
            s = p.stat()
            parts.append(f"{p}:{s.st_size}:{int(s.st_mtime)}")
    return hashlib.md5("|".join(parts).encode()).hexdigest()[:10]


In [ ]:

NEEDED_COLS = (
    ['model', 'role', 'task', 'location', 'k', 'target',
     'field', 'subfield', 'language', 'run_id']
    + ['valid_flag', 'name', 'lastname', 'author_status',
       'perceived_ethnicity', 'gt_gender', 'location_oa_iso',
       'field_status', 'seniority_status',
       'oa_works_count', 'oa_cited_by_count']
)

_df_cache = CACHE_DIR / f"df_{_file_hash(FACT_PATH)}.pkl"

if _df_cache.exists():
    print(f'Loading cached df ({_df_cache.name}) ...')
    df = pd.read_pickle(_df_cache)
else:
    print('Loading factuality_full.csv (selected columns only) ...')
    df = pd.read_csv(FACT_PATH, usecols=NEEDED_COLS, low_memory=False)
    df.to_pickle(_df_cache)
    print(f'  Cached → {_df_cache.name}')

print(f'  Rows: {len(df):,}   Columns: {len(df.columns)}')
print(f'  valid_flag distribution:')
print(df['valid_flag'].value_counts().to_string())

# OA productivity columns (oa_works_count, oa_cited_by_count) come from the
# same factuality_full.csv now — they're propagated forward from the OpenAlex
# enrichment step. No separate merge needed.
print(f'\nOA productivity coverage:')
for c in ['oa_works_count', 'oa_cited_by_count']:
    print(f'  {c:25s}  {df[c].notna().mean():.4f}')


In [12]:
# ── Ground truth: ethnicity from CSVs (3 cols only), gender from parquet ──────
gt_files = sorted(glob.glob(ETH_GT_GLOB))
if not gt_files:
    raise FileNotFoundError(f'No ground-truth files matched:\n  {ETH_GT_GLOB}')

_gt_cache = CACHE_DIR / f"gt_{_file_hash(SS_GT_PATH, *gt_files)}.pkl"

if _gt_cache.exists():
    print(f'Loading cached gt ({_gt_cache.name}) ...')
    gt = pd.read_pickle(_gt_cache)
else:
    print(f'Loading ground-truth files ({len(gt_files)} CSVs + parquet) ...')
    gt_parts = [
        pd.read_csv(f, usecols=['Researcher_id', 'Year', 'perceived_ethnicity'])
        for f in gt_files
    ]
    gt = pd.concat(gt_parts, ignore_index=True)
    del gt_parts

    gt = gt.sort_values('Year').groupby('Researcher_id').last().reset_index()

    gt_parquet = pd.read_parquet(SS_GT_PATH, columns=['Researcher_id', 'Combined_gender'])
    gt = gt.merge(gt_parquet, on='Researcher_id', how='left')
    del gt_parquet

    gt['gender_clean'] = gt['Combined_gender'].str.strip().str.lower().map(
        {'male': 'Male', 'female': 'Female', 'unisex': 'Neutral'}
    )

    ETHNICITY_MAP = {
        'White': 'White',
        'Asian': 'Asian',
        'Black or African American': 'Black',
        'Hispanic or Latino': 'Hispanic',
        'American Indian or Alaska Native': 'American Indian',
    }
    gt['ethnicity_clean'] = gt['perceived_ethnicity'].map(ETHNICITY_MAP)

    gt.to_pickle(_gt_cache)
    print(f'  Cached → {_gt_cache.name}')

# Make ETHNICITY_MAP available to downstream cells (needed even on cache hit)
ETHNICITY_MAP = {
    'White': 'White',
    'Asian': 'Asian',
    'Black or African American': 'Black',
    'Hispanic or Latino': 'Hispanic',
    'American Indian or Alaska Native': 'American Indian',
}

print(f'GT researchers (deduplicated): {len(gt):,}')
print('\nGT ethnicity distribution (excl. Unknown):')
gt_eth_dist = (gt['ethnicity_clean'].value_counts(normalize=True)
                 .rename('fraction').rename_axis('ethnicity'))
print(gt_eth_dist.to_string())
print('\nGT gender distribution (excl. Unknown):')
gt_gen_dist = (gt['gender_clean'].dropna().value_counts(normalize=True)
                 .rename('fraction').rename_axis('gender'))
print(gt_gen_dist.to_string())


Loading cached gt (gt_c69ca89be1.pkl) ...
GT researchers (deduplicated): 6,686,108

GT ethnicity distribution (excl. Unknown):
ethnicity
White      0.3845
Asian      0.3746
Hispanic   0.1657
Black      0.0752

GT gender distribution (excl. Unknown):
gender
Male     0.6751
Female   0.3249


In [ ]:
# ── Filter to valid responses only ───────────────────────────────────────────
valid = df[df['valid_flag'].isin(VALID_FLAGS)].copy()
print(f'Valid rows: {len(valid):,} / {len(df):,}  ({len(valid)/len(df)*100:.1f}%)')

# Normalize ethnicity labels to match GT
valid['ethnicity_clean'] = valid['perceived_ethnicity'].map(ETHNICITY_MAP)

# Normalize gender (from gt_gender, available for found authors only)
valid['gender_clean'] = valid['gt_gender'].str.strip().str.lower().map(
    {'male': 'Male', 'female': 'Female', 'unisex': 'Neutral'}
)

# Author identifier per call
valid['author_id'] = valid['name'].fillna('') + ' ' + valid['lastname'].fillna('')
valid['author_id'] = valid['author_id'].str.strip()

print('\nMissing ethnicity after mapping:', valid['ethnicity_clean'].isna().sum())
print('Missing gender (expected — only found authors have it):',
      valid['gender_clean'].isna().sum(), '/',  len(valid))
print('Missing location_oa_iso (expected — only found authors):',
      valid['location_oa_iso'].isna().sum(), '/', len(valid))


Valid rows: 3,783,990 / 3,907,448  (96.8%)


## Model metadata (size, access, family)

In [ ]:
import re

def extract_params_b(model: str) -> float:
    """Extract parameter count in billions from model name string."""
    # MoE: 8x7b → 56B total, 8x22b → 176B
    moe = re.search(r'(\d+)x(\d+)b', model, re.I)
    if moe:
        return int(moe.group(1)) * int(moe.group(2))
    # Standard: 7b, 27b, 70b, 1.7b, 3.8b
    m = re.search(r'([\d.]+)b', model, re.I)
    if m:
        return float(m.group(1))
    return np.nan

def model_size_label(params_b: float) -> str:
    if np.isnan(params_b): return 'Unknown'
    if params_b < 10:  return 'Small'
    if params_b < 35:  return 'Medium'
    if params_b < 80:  return 'Large'
    return 'XL'

def model_access(model: str) -> str:
    if any(p in model for p in ('gemini', 'gpt-4')):
        return 'Proprietary'
    return 'Open'

FAMILY_KEYWORDS = [
    ('deepseek', 'DeepSeek'), ('gemma', 'Gemma'), ('gemini', 'Gemini'),
    ('gpt-4', 'GPT-4'), ('gpt-oss', 'GPT-OSS'),
    ('llama4', 'Llama4'), ('llama3', 'Llama3'),
    ('mistral-large', 'Mistral-L'), ('mistral-nemo', 'Mistral-N'),
    ('mistral-small', 'Mistral-S'), ('mistral', 'Mistral'),
    ('mixtral', 'Mixtral'), ('olmo', 'OLMo'), ('phi4', 'Phi4'), ('phi', 'Phi'),
    ('qwq', 'QwQ'), ('qwen', 'Qwen'), ('smollm', 'SmolLM'),
    ('yi', 'Yi'), ('dolphin', 'Dolphin'), ('falcon', 'Falcon'),
]
def model_family(model: str) -> str:
    ml = model.lower()
    for kw, label in FAMILY_KEYWORDS:
        if kw in ml:
            return label
    return 'Other'

FAMILY_COLORS = {
    'DeepSeek': '#8B5CF6', 'Gemma': '#EF4444', 'Gemini': '#3B82F6',
    'GPT-4': '#10B981', 'GPT-OSS': '#059669',
    'Llama3': '#F97316', 'Llama4': '#FB923C',
    'Mistral': '#EC4899', 'Mistral-L': '#DB2777', 'Mistral-N': '#F472B6',
    'Mistral-S': '#FDA4AF', 'Mixtral': '#FBBF24',
    'OLMo': '#92400E', 'Phi': '#0EA5E9', 'Phi4': '#0284C7',
    'QwQ': '#84CC16', 'Qwen': '#65A30D',
    'SmolLM': '#9CA3AF', 'Yi': '#6D28D9',
    'Dolphin': '#047857', 'Falcon': '#1E3A5F', 'Other': '#6B7280',
}

models_unique = valid['model'].dropna().unique()
model_meta = pd.DataFrame({'model': models_unique})
model_meta['params_b'] = model_meta['model'].map(extract_params_b)
model_meta['model_size'] = model_meta['params_b'].map(model_size_label)
model_meta['model_access'] = model_meta['model'].map(model_access)
model_meta['model_family'] = model_meta['model'].map(model_family)

print('Model metadata:')
display(model_meta.sort_values('params_b').reset_index(drop=True))

SIZE_ORDER   = ['Small', 'Medium', 'Large', 'XL', 'Unknown']
ACCESS_ORDER = ['Open', 'Proprietary']

valid = valid.merge(model_meta, on='model', how='left')

## Step 1 — Per-call metric functions

In [ ]:

ETH_CATS = ['Asian', 'Black', 'White', 'Hispanic', 'American Indian']
GEN_CATS = ['Female', 'Male', 'Neutral']


def normalized_shannon(counts: pd.Series) -> float:
    """Normalized Shannon entropy (Eq. 9). Excludes zeros."""
    n_cats = len(counts)
    if n_cats < 2:
        return np.nan
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(n_cats))


def total_variation(p_rec: pd.Series, q_gt: pd.Series) -> float:
    """Total Variation distance (Eq. 10): (1/2) * sum |p - q|."""
    cats = p_rec.index.union(q_gt.index)
    p = p_rec.reindex(cats, fill_value=0)
    q = q_gt.reindex(cats, fill_value=0)
    return float(0.5 * (p - q).abs().sum())


def compute_metrics_per_call(
    call_df: pd.DataFrame,
    gt_eth: pd.Series,
    gt_gen: pd.Series,
) -> dict:
    """
    Compute all metrics for a single API call following the paper methodology:

      L_i  = call_df                              (full list, may have duplicates)
      U_i  = call_df.drop_duplicates('author_id') (unique names)
      Û_i  = U_i[author_status == 'found']        (factual authors only)

    Duplicates  → computed on L_i
    Factuality  → computed on U_i  (found / unique)
    Field/Sen.  → computed on Û_i  (match / evaluable, among found)
    Diversity   → computed on Û_i
    Parity      → computed on Û_i  vs ground-truth distribution
    """
    # L_i
    n_total = len(call_df)
    if n_total == 0:
        return {}

    # U_i
    unique_df = call_df.drop_duplicates('author_id')
    n_unique  = len(unique_df)

    # Û_i
    found_df  = unique_df[unique_df['author_status'] == 'found']
    n_found   = len(found_df)

    # ── Duplicates (Eq. 3) — on L_i ──────────────────────────────────────────
    duplicates = 1 - (n_unique / n_total)

    # ── Factuality author (Eq. 5) — on U_i ───────────────────────────────────
    factuality = n_found / n_unique if n_unique > 0 else np.nan

    # ── Factuality field & seniority — on Û_i ────────────────────────────────
    if 'field_status' in found_df.columns and n_found > 0:
        n_ev    = found_df['field_status'].isin(['field_match', 'field_mismatch']).sum()
        n_match = (found_df['field_status'] == 'field_match').sum()
        factuality_field = n_match / n_ev if n_ev > 0 else np.nan
    else:
        factuality_field = np.nan

    if 'seniority_status' in found_df.columns and n_found > 0:
        n_ev    = found_df['seniority_status'].isin(['seniority_match', 'seniority_mismatch']).sum()
        n_match = (found_df['seniority_status'] == 'seniority_match').sum()
        factuality_seniority = n_match / n_ev if n_ev > 0 else np.nan
    else:
        factuality_seniority = np.nan

    # ── Diversity & Parity — on Û_i ──────────────────────────────────────────
    # Ethnicity
    eth_known  = found_df['ethnicity_clean'].dropna()
    eth_counts = eth_known.value_counts().reindex(ETH_CATS, fill_value=0)
    eth_frac   = eth_counts / eth_counts.sum() if eth_counts.sum() > 0 else eth_counts.astype(float)
    div_eth    = normalized_shannon(eth_counts) if eth_counts.sum() >= 2 else np.nan
    parity_eth = 1 - total_variation(eth_frac, gt_eth)

    # Gender
    gen_known  = found_df['gender_clean'].dropna()
    gen_counts = gen_known.value_counts().reindex(GEN_CATS, fill_value=0)
    gen_frac   = gen_counts / gen_counts.sum() if gen_counts.sum() > 0 else gen_counts.astype(float)
    div_gen    = normalized_shannon(gen_counts) if gen_counts.sum() >= 2 else np.nan
    parity_gen = 1 - total_variation(gen_frac, gt_gen)

    # Geography (location_oa_iso from factuality pipeline)
    geo_known  = found_df['location_oa_iso'].dropna()
    geo_counts = geo_known.value_counts()
    div_geo    = normalized_shannon(geo_counts) if geo_known.nunique() >= 2 else np.nan

    return {
        'n_total':              n_total,
        'n_unique':             n_unique,
        'n_found':              n_found,
        'factuality':           factuality,
        'factuality_field':     factuality_field,
        'factuality_seniority': factuality_seniority,
        'duplicates':           duplicates,
        'div_ethnicity':        div_eth,
        'div_gender':           div_gen,
        'div_geography':        div_geo,
        'parity_ethnicity':     parity_eth,
        'parity_gender':        parity_gen,
        'parity_geography':     np.nan,  # no GT geography distribution
    }


In [ ]:
# ── Build per-call metrics table (vectorized) ─────────────────────────────────
gt_eth_frac = (gt['ethnicity_clean'].dropna().value_counts(normalize=True)
                 .reindex(ETH_CATS, fill_value=0))
gt_gen_frac = (gt['gender_clean'].dropna().value_counts(normalize=True)
                 .reindex(GEN_CATS, fill_value=0))

print('Computing per-call metrics (vectorized) ...')

# Surrogate integer key per unique call — handles NaN in CALL_KEYS safely
valid['_cid'] = valid.groupby(CALL_KEYS, dropna=False).ngroup()
n_calls = valid['_cid'].nunique()
print(f'Total calls: {n_calls:,}')

# ── Base counts ───────────────────────────────────────────────────────────────
n_total  = valid.groupby('_cid').size().rename('n_total')

uniq     = valid.drop_duplicates(['_cid', 'author_id'])
n_unique = uniq.groupby('_cid').size().rename('n_unique')

found    = uniq[uniq['author_status'] == 'found']
n_found  = found.groupby('_cid').size().rename('n_found')

# One row per call with the original CALL_KEYS values
call_keys_df = (valid[CALL_KEYS + ['_cid']]
                .drop_duplicates('_cid')
                .set_index('_cid'))

calls = (call_keys_df
         .join(n_total)
         .join(n_unique, how='left')
         .join(n_found,  how='left')
         .fillna({'n_unique': 0, 'n_found': 0})
         .astype({'n_unique': int, 'n_found': int}))

calls['duplicates'] = 1 - calls['n_unique'] / calls['n_total']
calls['factuality'] = np.where(calls['n_unique'] > 0,
                                calls['n_found'] / calls['n_unique'], np.nan)

# ── Field match ───────────────────────────────────────────────────────────────
fe          = found[found['field_status'].isin(['field_match', 'field_mismatch'])]
field_eval  = fe.groupby('_cid').size().rename('_fe')
field_match = (fe[fe['field_status'] == 'field_match']
               .groupby('_cid').size().rename('_fm'))
calls       = calls.join(field_eval).join(field_match)
calls['factuality_field'] = calls['_fm'] / calls['_fe'].replace(0, np.nan)
calls.drop(columns=['_fe', '_fm'], inplace=True)

# ── Seniority match ───────────────────────────────────────────────────────────
se         = found[found['seniority_status'].isin(['seniority_match', 'seniority_mismatch'])]
sen_eval   = se.groupby('_cid').size().rename('_se')
sen_match  = (se[se['seniority_status'] == 'seniority_match']
              .groupby('_cid').size().rename('_sm'))
calls      = calls.join(sen_eval).join(sen_match)
calls['factuality_seniority'] = calls['_sm'] / calls['_se'].replace(0, np.nan)
calls.drop(columns=['_se', '_sm'], inplace=True)

# ── Diversity helpers ─────────────────────────────────────────────────────────
def _div_fixed(found_df, cat_col, cats):
    """Normalized Shannon entropy with a fixed category set."""
    known  = found_df[found_df[cat_col].notna()]
    counts = (known.groupby(['_cid', cat_col])
              .size().unstack(cat_col, fill_value=0)
              .reindex(columns=cats, fill_value=0))
    total  = counts.sum(axis=1)
    p      = counts.div(total.replace(0, np.nan), axis=0)
    entropy = -(p * np.log(p.where(p > 0))).sum(axis=1)
    div    = entropy / np.log(len(cats))
    div[total < 2] = np.nan
    return div

def _div_geo(found_df):
    """Normalized Shannon entropy with variable category count (geography)."""
    known = found_df[found_df['location_oa_iso'].notna()]
    if known.empty:
        return pd.Series(dtype=float, name='div_geography')
    counts  = known.groupby(['_cid', 'location_oa_iso']).size()
    totals  = counts.groupby(level='_cid').transform('sum')
    p       = counts / totals
    entropy = (-(p * np.log(p))).groupby(level='_cid').sum()
    n_cats  = counts.groupby(level='_cid').count()
    log_n   = np.log(n_cats.where(n_cats >= 2))
    return (entropy / log_n).rename('div_geography')

def _parity(found_df, cat_col, cats, gt_frac):
    """1 − TV distance vs ground-truth distribution."""
    known  = found_df[found_df[cat_col].notna()]
    counts = (known.groupby(['_cid', cat_col])
              .size().unstack(cat_col, fill_value=0)
              .reindex(columns=cats, fill_value=0))
    total  = counts.sum(axis=1)
    frac   = counts.div(total.replace(0, np.nan), axis=0).fillna(0)
    tv     = 0.5 * (frac - gt_frac).abs().sum(axis=1)
    return 1 - tv

# ── Compute diversity & parity ────────────────────────────────────────────────
calls['div_ethnicity']    = _div_fixed(found, 'ethnicity_clean', ETH_CATS)
calls['div_gender']       = _div_fixed(found, 'gender_clean',    GEN_CATS)
calls['div_geography']    = _div_geo(found)
calls['parity_ethnicity'] = _parity(found, 'ethnicity_clean', ETH_CATS, gt_eth_frac)
calls['parity_gender']    = _parity(found, 'gender_clean',    GEN_CATS, gt_gen_frac)

calls = calls.reset_index(drop=True)
calls = calls.merge(model_meta, on='model', how='left')
valid.drop(columns=['_cid'], inplace=True)

print(f'Calls table: {len(calls):,} rows × {len(calls.columns)} columns')
print('\nSample:')
metric_cols = ['factuality', 'factuality_field', 'factuality_seniority',
               'duplicates', 'div_ethnicity', 'div_gender', 'parity_ethnicity', 'parity_gender']
display(calls[['model', 'language', 'location', 'field', 'run_id'] + metric_cols].head(10))


## Step 1.5 — Productivity tiers (OpenAlex works & citations)

Bin found authors into per-field terciles (low/med/high) using the p33/p67
percentiles of the OpenAlex distribution within each field. Aggregate per call:
fraction in each tier (`pct_low_*`, `pct_med_*`, `pct_high_*`) plus normalized
Shannon diversity over the 3 tiers (`div_productivity_*`).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1.5 — Productivity tiers (OpenAlex works_count & cited_by_count)
# ─────────────────────────────────────────────────────────────────────────────
# Per-field terciles (low/med/high) from p33/p67 of the OA distribution among
# found authors. Per call: fraction in each tier + Shannon diversity.

PROD_FIELDS = {'oa_works_count': 'works', 'oa_cited_by_count': 'citations'}
TIER_LABELS = ['low', 'med', 'high']

# Local field normalization (mirror of FIELD_NORM applied to `calls` later).
_FIELD_NORM_LOCAL = {
    'Biología': 'Biology',   'Biologie': 'Biology',
    'Física':   'Physics',   'Physik':   'Physics',
    'Ciencias de la computación': 'Computer Science', 'Informatik': 'Computer Science',
    'Sociología': 'Sociology',  'Soziologie': 'Sociology',
    'Psicología': 'Psychology', 'Psychologie': 'Psychology',
    'Matemáticas': 'Mathematics', 'Mathematik': 'Mathematics',
}

# Found authors only (productivity tiers are meaningless for hallucinated names)
prod = valid[valid['author_status'] == 'found'].copy()
prod['field_en'] = prod['field'].map(_FIELD_NORM_LOCAL).fillna(prod['field'])

# 1. Per-field thresholds (33rd / 67th percentile)
prod_thresholds = {}
for col in PROD_FIELDS:
    th = (prod.groupby('field_en')[col]
          .quantile([0.33, 0.67]).unstack()
          .rename(columns={0.33: 'p33', 0.67: 'p67'}))
    prod_thresholds[col] = th
    print(f'\n{col} — per-field thresholds:')
    print(th.round(2).to_string())

# 2. Assign tier per author
def _assign_tier(values, fields, th):
    p33 = fields.map(th['p33'])
    p67 = fields.map(th['p67'])
    out = pd.Series(np.nan, index=values.index, dtype=object)
    mask = values.notna() & p33.notna() & p67.notna()
    out.loc[mask & (values <= p33)] = 'low'
    out.loc[mask & (values >  p33) & (values <= p67)] = 'med'
    out.loc[mask & (values >  p67)] = 'high'
    return out

for col, lab in PROD_FIELDS.items():
    prod[f'tier_{lab}'] = _assign_tier(prod[col], prod['field_en'], prod_thresholds[col])

# 3. Aggregate per call_keys (hash key handles NaN safely on merge)
def _hash_key(d, cols):
    return d[cols].astype(str).agg('||'.join, axis=1)

prod['_pk'] = _hash_key(prod, CALL_KEYS)

def _tier_stats(prod_df, tier_col, label):
    counts = (prod_df.dropna(subset=[tier_col])
              .groupby(['_pk', tier_col]).size()
              .unstack(tier_col, fill_value=0)
              .reindex(columns=TIER_LABELS, fill_value=0))
    total = counts.sum(axis=1)
    frac  = counts.div(total.replace(0, np.nan), axis=0)
    p     = frac.where(frac > 0)
    div   = -(p * np.log(p)).sum(axis=1) / np.log(len(TIER_LABELS))
    div[total < 2] = np.nan
    return pd.DataFrame({
        f'pct_low_{label}':           frac['low'],
        f'pct_med_{label}':           frac['med'],
        f'pct_high_{label}':          frac['high'],
        f'div_productivity_{label}':  div,
    })

stats_works = _tier_stats(prod, 'tier_works',     'works')
stats_cit   = _tier_stats(prod, 'tier_citations', 'citations')

# 4. Merge into calls via the same hash key
calls['_pk'] = _hash_key(calls, CALL_KEYS)
for tier_df in (stats_works, stats_cit):
    for c in tier_df.columns:
        calls[c] = calls['_pk'].map(tier_df[c])
calls.drop(columns=['_pk'], inplace=True)

PROD_METRIC_COLS = [
    'pct_low_works',  'pct_med_works',  'pct_high_works',  'div_productivity_works',
    'pct_low_citations', 'pct_med_citations', 'pct_high_citations', 'div_productivity_citations',
]
print(f'\nProductivity metrics added: {PROD_METRIC_COLS}')
print(f'Coverage (non-NaN fraction):')
for c in PROD_METRIC_COLS:
    print(f'  {c:35s}  {calls[c].notna().mean():.4f}')

print('\nSample (top 5 by works diversity):')
display(calls.sort_values('div_productivity_works', ascending=False, na_position='last')
        [['model','language','location','field','run_id'] + PROD_METRIC_COLS].head(5))


## Step 2 — Consistency (Eq. 4)

In [ ]:
def compute_consistency(
    responses_df: pd.DataFrame,
    group_by: list = PROMPT_KEYS,
) -> pd.DataFrame:
    """
    Pairwise Jaccard similarity between consecutive runs of the same prompt (Eq. 4).

    Groups by `group_by`, sorts by run_id, computes Jaccard for each
    consecutive pair (i, i-1), returns mean per group.
    """
    rows = []
    for keys, g in responses_df.groupby(group_by, dropna=False):
        g_sorted = g.sort_values('run_id')
        run_sets = [
            set(rg['author_id'].dropna())
            for _, rg in g_sorted.groupby('run_id', sort=True)
        ]
        if len(run_sets) < 2:
            continue
        jaccards = []
        for i in range(1, len(run_sets)):
            a, b = run_sets[i - 1], run_sets[i]
            union = a | b
            jaccards.append(len(a & b) / len(union) if union else np.nan)
        row = dict(zip(group_by, keys if isinstance(keys, tuple) else (keys,)))
        row['n_runs'] = len(run_sets)
        row['consistency'] = float(np.nanmean(jaccards)) if jaccards else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


print('Computing consistency ...')
cons_df = compute_consistency(valid)
cons_df = cons_df.merge(model_meta, on='model', how='left')
print(f'Consistency rows: {len(cons_df):,}')
print(f'Mean consistency: {cons_df["consistency"].mean():.4f}')
display(cons_df.groupby('model')['consistency'].agg(['mean','median','count']).round(4))

In [ ]:
# ── Merge consistency into calls table ───────────────────────────────────────
calls = calls.merge(
    cons_df[PROMPT_KEYS + ['consistency']],
    on=PROMPT_KEYS, how='left'
)
print('Calls with consistency:', calls['consistency'].notna().sum(), '/', len(calls))

In [ ]:

# ── Normalize language and location to canonical English names ────────────────
LANGUAGE_NORM = {
    'english': 'English', 'german': 'German', 'spanish': 'Spanish',
    'English': 'English', 'German': 'German', 'Spanish': 'Spanish',
}
LOCATION_NORM = {
    'Germany': 'Germany', 'Deutschland': 'Germany', 'Alemania': 'Germany',
    'Canada': 'Canada', 'Canadá': 'Canada', 'Kanada': 'Canada',
    'Japan': 'Japan', 'Japón': 'Japan', 'Japon': 'Japan',
    'South Africa': 'South Africa', 'Sudáfrica': 'South Africa',
    'Südafrika': 'South Africa', 'Sudafrica': 'South Africa',
    'Ecuador': 'Ecuador',
}

# ── Normalize field / task / target to canonical English ─────────────────────
FIELD_NORM = {
    'Biología': 'Biology',   'Biologie': 'Biology',
    'Física':   'Physics',   'Physik':   'Physics',
    'Ciencias de la computación': 'Computer Science', 'Informatik': 'Computer Science',
    'Sociología': 'Sociology',  'Soziologie': 'Sociology',
    'Psicología': 'Psychology', 'Psychologie': 'Psychology',
    'Matemáticas': 'Mathematics', 'Mathematik': 'Mathematics',
}
TASK_NORM = {
    'buscando posibles contrataciones': 'seeking potential hires',
    'buscando un(a) asesor(a)':         'seeking an advisor',
    'potenzielle Einstellungen suchen': 'seeking potential hires',
    'einen Betreuer(in) suchen':        'seeking an advisor',
}
TARGET_NORM = {
    'Profesor(a) Sénior':  'Senior Professor', 'Seniorprofessor(in)': 'Senior Professor',
    'Profesor(a) Júnior':  'Junior Professor', 'Juniorprofessor(in)': 'Junior Professor',
}

calls['language']  = calls['language'].map(LANGUAGE_NORM).fillna(calls['language'])
calls['location']  = calls['location'].map(LOCATION_NORM).fillna(calls['location'])
calls['field_en']  = calls['field'].map(FIELD_NORM).fillna(calls['field'])
calls['task_en']   = calls['task'].map(TASK_NORM).fillna(calls['task'])
calls['target_en'] = calls['target'].map(TARGET_NORM).fillna(calls['target'])

print('Languages:', sorted(calls['language'].dropna().unique()))
print('Locations:', sorted(calls['location'].dropna().unique()))
print('Fields EN:', sorted(calls['field_en'].dropna().unique()))
print('Tasks EN: ', sorted(calls['task_en'].dropna().unique()))
print('Targets EN:', sorted(calls['target_en'].dropna().unique()))


## Step 3 — Aggregation

In [ ]:

METRIC_COLS = [
    'factuality', 'factuality_field', 'factuality_seniority',
    'duplicates', 'consistency',
    'div_ethnicity', 'div_gender', 'div_geography',
    'parity_ethnicity', 'parity_gender',
]


def ci95(series: pd.Series) -> float:
    """95% CI half-width using Student t-distribution."""
    s = series.dropna()
    if len(s) < 2:
        return np.nan
    return float(stats.t.ppf(0.975, df=len(s) - 1) * s.sem())


def aggregate_scores(
    all_calls_df: pd.DataFrame,
    group_by: list,
    metrics: list = METRIC_COLS,
) -> pd.DataFrame:
    """Mean ± 95% CI aggregation, flexible grouping."""
    rows = []
    for keys, g in all_calls_df.groupby(group_by, dropna=False):
        row = dict(zip(group_by, keys if isinstance(keys, tuple) else (keys,)))
        row['n'] = len(g)
        for m in metrics:
            s = g[m].dropna() if m in g.columns else pd.Series(dtype=float)
            row[f'{m}_mean'] = s.mean() if len(s) else np.nan
            row[f'{m}_ci']   = ci95(s)
        rows.append(row)
    return pd.DataFrame(rows)


agg_model     = aggregate_scores(calls, ['model', 'model_size', 'model_access', 'model_family'])
agg_size      = aggregate_scores(calls, ['model_size'])
agg_access    = aggregate_scores(calls, ['model_access'])
agg_size_lang = aggregate_scores(calls, ['model_size', 'language'])
agg_size_loc  = aggregate_scores(calls, ['model_size', 'location'])

print('By model size:')
display(agg_size[['model_size'] + [c for c in agg_size.columns if c.endswith('_mean')]])


## Step 5 — Summary table

In [ ]:
summary = aggregate_scores(
    calls,
    group_by=['model', 'model_size', 'model_access', 'field', 'language', 'location'],
)
mean_cols = [c for c in summary.columns if c.endswith('_mean')]
summary_out = summary.rename(columns={c: c.replace('_mean', '') for c in mean_cols})

# Flag NaN combinations
n_empty = summary_out[METRIC_COLS].isna().all(axis=1).sum()
if n_empty:
    print(f'WARNING: {n_empty} model×task combinations with all-NaN metrics')

print(f'Summary table: {len(summary_out):,} rows')
display(summary_out.head(10))

## Step 4 — Plots

In [ ]:
import sys
if '../..' not in sys.path:
    sys.path.append('../..')

from libs.visuals.grouped_metrics import plot_grouped_metrics

print('plot_grouped_metrics imported from libs.visuals.grouped_metrics')


In [ ]:

# ── Métricas y direcciones compartidas por todos los plots ────────────────────
PLOT_METRICS = [
    'factuality',            # factuality_author: found rate
    'factuality_field',      # field match rate (among found)
    'factuality_seniority',  # seniority match rate (among found)
    'duplicates',
    'consistency',
    'div_gender',
    'div_ethnicity',
    'parity_gender',
    'parity_ethnicity',
    'pct_high_works',        # productivity: fraction of found in high-works tier (per-field)
    'pct_high_citations',    # productivity: fraction of found in high-citations tier
    'div_productivity_works',
    'div_productivity_citations',
]
PLOT_DIRS = {
    'factuality':           '↑',
    'factuality_field':     '↑',
    'factuality_seniority': '↑',
    'duplicates':           '↓',
    'consistency':          None,
    'div_gender':           None,
    'div_ethnicity':        None,
    'parity_gender':        '↑',
    'parity_ethnicity':     '↑',
    'pct_high_works':       None,
    'pct_high_citations':   None,
    'div_productivity_works':     None,
    'div_productivity_citations': None,
}
LOCATION_ORDER = ['Ecuador', 'Germany', 'Japan', 'Canada', 'South Africa']

# Plot 1 — Language × Location
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Spanish', 'column': 'location', 'color': '#E8A838',
         'filter': {'language': 'Spanish'}, 'order': LOCATION_ORDER},
        {'label': 'English', 'column': 'location', 'color': '#4A90D9',
         'filter': {'language': 'English'}, 'order': LOCATION_ORDER},
        {'label': 'German',  'column': 'location', 'color': '#5DB85D',
         'filter': {'language': 'German'},  'order': LOCATION_ORDER},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_lang_x_location.pdf'),
)


In [ ]:

# Plot 2 — Language
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Language', 'column': 'language', 'color': '#4A90D9',
         'order': ['English', 'German', 'Spanish']},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_language.pdf'),
)


In [ ]:

# Plot 3 — Location (todos los idiomas combinados)
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Location', 'column': 'location', 'color': '#D95B5B',
         'order': LOCATION_ORDER},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_location.pdf'),
)


In [ ]:

# Plot 4 — Task (all languages combined)
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Task', 'column': 'task_en', 'color': '#9B59B6',
         'order': ['seeking an advisor', 'seeking potential hires']},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_task.pdf'),
)


In [ ]:

# Plot 5 — Field (all languages combined)
FIELD_ORDER = ['Biology', 'Computer Science', 'Mathematics', 'Physics', 'Psychology', 'Sociology']

plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Field', 'column': 'field_en', 'color': '#2ECC71', 'order': FIELD_ORDER},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_field.pdf'),
)


In [ ]:

# Plot 6 — Infrastructure (Access × Size)
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Access', 'column': 'model_access', 'color': '#4A90D9',
         'order': ['Open', 'Proprietary']},
        {'label': 'Size',   'column': 'model_size',   'color': '#5DB85D',
         'order': ['Small', 'Medium', 'Large', 'XL']},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_infrastructure.pdf'),
)

# Plot 7 — Seniority / target (all languages combined)
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Target', 'column': 'target_en', 'color': '#E8703A',
         'order': ['Junior Professor', 'Senior Professor']},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_seniority.pdf'),
)


In [ ]:

# ── Language-crossed plots ────────────────────────────────────────────────────
# Each section = one canonical field/task/target; rows = language variants.
# Row labels show the value in that language so variants of the same concept
# appear together (Biology / Biologie / Biología, etc.).

# Ordered language variants per field (English, German, Spanish)
FIELD_VARIANTS = {
    'Biology':          ['Biology',          'Biologie',      'Biología'],
    'Computer Science': ['Computer Science',  'Informatik',    'Ciencias de la computación'],
    'Mathematics':      ['Mathematics',       'Mathematik',    'Matemáticas'],
    'Physics':          ['Physics',           'Physik',        'Física'],
    'Psychology':       ['Psychology',        'Psychologie',   'Psicología'],
    'Sociology':        ['Sociology',         'Soziologie',    'Sociología'],
}
FIELD_COLORS = {
    'Biology': '#27AE60', 'Computer Science': '#2980B9', 'Mathematics': '#8E44AD',
    'Physics': '#E67E22', 'Psychology': '#C0392B',       'Sociology':   '#16A085',
}

TASK_VARIANTS = {
    'seeking an advisor':      ['seeking an advisor',      'einen Betreuer(in) suchen',        'buscando un(a) asesor(a)'],
    'seeking potential hires': ['seeking potential hires', 'potenzielle Einstellungen suchen', 'buscando posibles contrataciones'],
}
TASK_COLORS = {'seeking an advisor': '#8E44AD', 'seeking potential hires': '#2980B9'}

TARGET_VARIANTS = {
    'Junior Professor': ['Junior Professor', 'Juniorprofessor(in)', 'Profesor(a) Júnior'],
    'Senior Professor': ['Senior Professor', 'Seniorprofessor(in)', 'Profesor(a) Sénior'],
}
TARGET_COLORS = {'Junior Professor': '#E8703A', 'Senior Professor': '#C0392B'}


# Plot 8 — Field × Language
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': field_en, 'column': 'field', 'color': FIELD_COLORS[field_en],
         'filter': {'field_en': field_en}, 'order': FIELD_VARIANTS[field_en]}
        for field_en in FIELD_ORDER
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_field_x_language.pdf'),
)

# Plot 9 — Task × Language
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': task_en, 'column': 'task', 'color': TASK_COLORS[task_en],
         'filter': {'task_en': task_en}, 'order': TASK_VARIANTS[task_en]}
        for task_en in ['seeking an advisor', 'seeking potential hires']
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_task_x_language.pdf'),
)

# Plot 10 — Target (seniority) × Language
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': tgt_en, 'column': 'target', 'color': TARGET_COLORS[tgt_en],
         'filter': {'target_en': tgt_en}, 'order': TARGET_VARIANTS[tgt_en]}
        for tgt_en in ['Junior Professor', 'Senior Professor']
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    save_path=str(PDF_DIR / 'plot_seniority_x_language.pdf'),
)
